In [0]:
%pip install ftfy>=6.3 mammoth>=1.7 resiliparse>=0.14 pypdfium2>=4.30 loky
%pip install striprtf ocflzw-decompress  # existing deps, ensure installed
%pip install pytesseract Pillow PyMuPDF  # OCR cascade deps (tesseract binary provided by cluster init script)
dbutils.library.restartPython()

In [0]:
# Pre-flight: fail fast if cluster lacks required system binaries or Python libs.
import subprocess
_preflight_errors = []
try:
    subprocess.run(["tesseract", "--version"], check=True, capture_output=True, timeout=10)
except Exception as e:
    _preflight_errors.append(f"tesseract missing or broken: {e}")
for mod in ("ftfy", "mammoth", "resiliparse", "pypdfium2", "striprtf"):
    try:
        __import__(mod)
    except Exception as e:
        _preflight_errors.append(f"cannot import {mod}: {e}")
if _preflight_errors:
    raise RuntimeError("Pre-flight failed:\n" + "\n".join(_preflight_errors))
print("Pre-flight OK")

In [0]:
# Shared extraction library for the blob processing pipeline (v2).
# %run-imported by blob_processing_v2, blob_replay, blob_validation_harness,
# blob_lzw_port_tests.
#
# DO NOT call spark.sql or write any tables from this notebook.
# It defines pure functions and constants only.

import hashlib, io, json, os, re, time
from typing import Optional, Tuple

# ==================== VERSION CONSTANTS ====================
# Bump exactly one of these when the corresponding stage changes.
# Bumping invalidates all rows with a lower version; the replay notebook
# will re-run the affected stage when its scope predicate matches.

DECOMPRESSOR_VERSION = 2
PARSER_VERSION = 2
POST_PROCESSOR_VERSION = 3

# ==================== SIZE LIMITS ====================
MAX_BLOB_SIZE = 16 * 1024 * 1024  # 16 MB; rows above this are flagged "Compressed Too Large"
OCR_MAX_PAGES = 10
OCR_MAX_PDF_SIZE_MB = 50
OCR_LANG = "eng"
# pytesseract's `timeout=` is forwarded to the tesseract subprocess and actually
# kills it (not a Python-side abort). Prevents pathological PDFs from hanging.
OCR_PAGE_TIMEOUT_SEC = 30

# PDF hang defenses — see ParsePool below.
# Size cap skips decompression-DoS-scale inputs before we ever touch pypdfium2 /
# PyMuPDF; the subprocess timeout handles small-but-pathological PDFs that hang
# native code paths which have no Python-side timeout hook.
PDF_EXTRACT_MAX_BYTES = 20 * 1024 * 1024  # 20 MB — skip parsing, mark as "[PDF - too large]"
PARSE_TIMEOUT_SEC = 30  # hard wall-clock cap per parse() call, enforced via subprocess

# ==================== BYTE MARKERS ====================
OCF_MARKER = b'ocf_blob\0'
PDF_MAGIC = b'%PDF-'
RTF_MAGIC = b'{\\rtf'
RTF_MAGIC_LOOSE = b'{\\'
ZIP_MAGIC = b'\x50\x4B\x03\x04'
OLE_MAGIC = b'\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1'
TIFF_LE_MAGIC = b'II*\x00'
TIFF_BE_MAGIC = b'MM\x00*'

CONTAINER_MAGICS = [PDF_MAGIC, RTF_MAGIC, ZIP_MAGIC, OLE_MAGIC, TIFF_LE_MAGIC, TIFF_BE_MAGIC]

# ==================== BINARY CONTENT TYPES ====================
BINARY_CONTENT_TYPES = frozenset({
    'image/tiff', 'image/png', 'image/g3fax', 'image/x-tga', 'image/jpeg',
    'application/x-dosexec', 'application/x-matlab-data', 'application/x-coff',
    'audio/x-mp4a-latm', 'audio/mpeg',
    'application/zlib', 'application/x-lzma', 'application/x-compress',
    'application/SIMH-tape-data', 'font/x-amiga-font',
    'application/x-executable', 'application/x-sharedlib',
})

# ==================== SANITY LIMITS ====================
BLOB_LENGTH_TOLERANCE = 0.01  # decompressed length within 1% of BLOB_LENGTH before flagging truncation
LZW_LOOP_BUDGET_MULT = 16     # max LZW iterations = compressed_size * this

# ==================== CONTENT HASHING ====================

def raw_content_sha256(chunks: list) -> str:
    """Stable hash of concatenated raw chunk bytes (in BLOB_SEQ_NUM order).

    Input: list of (BLOB_SEQ_NUM, BLOB_CONTENTS) tuples or dict-like rows.
    Output: 64-char hex digest.
    """
    h = hashlib.sha256()
    for c in sorted(chunks, key=lambda x: x['BLOB_SEQ_NUM'] or 0):
        contents = c['BLOB_CONTENTS']
        if contents:
            h.update(contents)
    return h.hexdigest()

In [0]:
# ==================== CERNER LZW DECODER (pgodwin-port) ====================
# Python port of pgodwin/OcfLzw2/LZWDecoder.cs with three hardening changes:
#   1. Growable bytearray output (replaces fixed 8192 buffer)
#   2. Output bytes masked with & 0xFF
#   3. Loop-iteration budget instead of signal-based timeout

class CernerLZWError(Exception):
    """Raised by cerner_lzw_decompress on unrecoverable input."""


def cerner_lzw_decompress(data: bytes, *, loop_budget: Optional[int] = None, max_output_bytes: int = 256 * 1024 * 1024) -> bytes:
    """Decompress Cerner OCF LZW bytes.

    Format: MSB-first bit packing; 9 -> 13 bit variable code width; clear=256;
    EOF=257; code width bumps at code boundaries 511, 1023, 2047, 4095
    (early-change: width increases one code before the dictionary fills that
    width, matching the Cerner encoder).

    Args:
        data: Raw compressed bytes with OCF wrapper already stripped.
        loop_budget: Max iterations; defaults to len(data) * LZW_LOOP_BUDGET_MULT.
        max_output_bytes: Max output size; defaults to 256 MiB.

    Returns:
        Decompressed bytes.

    Raises:
        CernerLZWError: on malformed input that cannot be recovered.
    """
    if not data:
        return b""

    CLEAR_CODE = 256
    EOI_CODE = 257
    FIRST_CODE = 258
    MAX_CODE = (1 << 13) - 1  # 8191; pgodwin uses 13-bit max dictionary

    dictionary = {i: bytes([i & 0xFF]) for i in range(256)}
    next_code = FIRST_CODE
    code_size = 9

    output = bytearray()

    def _check_output_size():
        if len(output) > max_output_bytes:
            raise CernerLZWError(f"output size {len(output)} exceeds max {max_output_bytes}")

    prev_string = b""

    # MSB-first bit buffer
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        code = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return code

    if loop_budget is None:
        loop_budget = max(1024, data_len * LZW_LOOP_BUDGET_MULT)

    iterations = 0

    # Read first code (allowing a leading clear)
    code = read_code()
    if code is None:
        return bytes(output)
    if code == CLEAR_CODE:
        code = read_code()
        if code is None or code == EOI_CODE:
            return bytes(output)

    if code < 256:
        output.append(code & 0xFF)
        _check_output_size()
        prev_string = bytes([code & 0xFF])
    elif code in dictionary:
        output.extend(dictionary[code])
        _check_output_size()
        prev_string = dictionary[code]
    else:
        raise CernerLZWError(f"first code {code} is not a literal or dictionary entry")

    while iterations < loop_budget:
        iterations += 1
        code = read_code()
        if code is None or code == EOI_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i & 0xFF]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            prev_string = b""
            code = read_code()
            if code is None or code == EOI_CODE:
                break
            if code < 256:
                output.append(code & 0xFF)
                _check_output_size()
                prev_string = bytes([code & 0xFF])
            elif code in dictionary:
                output.extend(dictionary[code])
                _check_output_size()
                prev_string = dictionary[code]
            else:
                raise CernerLZWError(f"post-clear code {code} invalid")
            continue

        if code < 256:
            current_string = bytes([code & 0xFF])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            # KwKwK
            current_string = prev_string + prev_string[0:1]
        else:
            raise CernerLZWError(f"unknown code {code} at iter {iterations} "
                                 f"(next_code={next_code}, dict_size={len(dictionary)})")

        output.extend(current_string)
        _check_output_size()

        if prev_string and next_code <= MAX_CODE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            # Early-change width bumps: width increases one code before the
            # dictionary fills that width (matches the Cerner encoder output
            # and the working TIFF-variant fallback below). The previous
            # late-change thresholds (512/1024/2048/4096) were off-by-one
            # and caused bit-stream desync at each width boundary.
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        prev_string = current_string
    else:
        raise CernerLZWError(f"loop budget {loop_budget} exceeded")

    return bytes(output)

In [0]:
# ==================== OCF WRAPPER + MAGIC SNIFF ====================

def strip_ocf_aggressive(data: bytes) -> bytes:
    if not data:
        return data
    if data.endswith(OCF_MARKER):
        data = data[:-len(OCF_MARKER)]
    if OCF_MARKER in data:
        data = b"".join(data.split(OCF_MARKER))
    return data


def strip_ocf_conservative(data: bytes) -> bytes:
    if not data:
        return data
    if data.endswith(OCF_MARKER):
        return data[:-len(OCF_MARKER)]
    return data


def sniff_container_magic(data: bytes) -> bool:
    """True if data begins with a known container magic (PDF/RTF/ZIP/OLE/TIFF)."""
    if not data:
        return False
    head = data[:16]
    if head.startswith(RTF_MAGIC):
        return True
    for magic in (PDF_MAGIC, ZIP_MAGIC, OLE_MAGIC, TIFF_LE_MAGIC, TIFF_BE_MAGIC):
        if head.startswith(magic):
            return True
    # loose RTF: '{\\' — only trust if followed by an ASCII letter
    if head.startswith(RTF_MAGIC_LOOSE) and len(head) > 2 and 0x20 <= head[2] < 0x7F:
        return True
    return False

In [0]:
# ==================== FALLBACK LZW DECODERS (ported from v1) ====================

def tolerant_lzw_decompress(data: bytes) -> bytes:
    """LSB-first tolerant decoder — fallback for blobs the pgodwin port can't handle.
    Carried over from blob_processing v1 with minor cleanup.
    """
    if not data:
        return b""
    CLEAR_CODE, END_CODE, FIRST_CODE, MAX_CODE = 256, 257, 258, 65535
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code = FIRST_CODE
    output = bytearray()
    prev_string = b""
    code_size = 9
    max_code_for_size = 512
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)
    max_iters = data_len * 8
    iters = 0
    while iters < max_iters:
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer |= data[byte_pos] << bits_in_buffer
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            break
        code = bit_buffer & ((1 << code_size) - 1)
        bit_buffer >>= code_size
        bits_in_buffer -= code_size
        iters += 1
        if code == END_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            max_code_for_size = 512
            prev_string = b""
            continue
        if code < 256:
            current_string = bytes([code])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            current_string = prev_string + prev_string[0:1]
        else:
            if prev_string:
                current_string = prev_string + prev_string[0:1]
            else:
                continue
        output.extend(current_string)
        if prev_string and next_code <= MAX_CODE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            if next_code >= max_code_for_size and code_size < 16:
                code_size += 1
                max_code_for_size *= 2
        prev_string = current_string
    return bytes(output)


def tiff_lzw_decompress(data: bytes) -> bytes:
    """TIFF-variant LZW (MSB-first with early-change quirk). 8192-entry max.
    Carried over from blob_processing v1 as a third-tier fallback.
    """
    if not data:
        return b""
    CLEAR_CODE, EOI_CODE, FIRST_CODE, MAX_DICT_SIZE = 256, 257, 258, 8192
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code = FIRST_CODE
    code_size = 9
    output = bytearray()
    prev_string = b""
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        c = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return c

    code = read_code()
    if code is None:
        return bytes(output)
    if code == CLEAR_CODE:
        code = read_code()
        if code is None or code == EOI_CODE:
            return bytes(output)
    if code < 256:
        output.append(code)
        prev_string = bytes([code])
    elif code in dictionary:
        output.extend(dictionary[code])
        prev_string = dictionary[code]
    else:
        return bytes(output)

    max_iters = data_len * 10
    for _ in range(max_iters):
        code = read_code()
        if code is None or code == EOI_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            prev_string = b""
            code = read_code()
            if code is None or code == EOI_CODE:
                break
            if code < 256:
                output.append(code)
                prev_string = bytes([code])
            elif code in dictionary:
                output.extend(dictionary[code])
                prev_string = dictionary[code]
            continue
        if code < 256:
            current_string = bytes([code])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            current_string = prev_string + prev_string[0:1]
        else:
            continue
        output.extend(current_string)
        if prev_string and next_code < MAX_DICT_SIZE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        prev_string = current_string
    return bytes(output)

In [0]:
# ==================== DECOMPRESS DISPATCHER ====================

try:
    from ocflzw_decompress.lzw import LzwDecompress
    _HAVE_OCFLZW = True
except ImportError:
    _HAVE_OCFLZW = False


def decompress(raw: bytes, compression_cd, chunk_count: int, blob_length: Optional[int], metrics: dict):
    """Main decompression entry point.

    Sets these keys in `metrics`:
        decompression_strategy, ocf_marker_count, blob_length_mismatch,
        blob_length_expected, blob_length_actual
    Returns (decompressed_bytes, error_message).
    On success error_message is None. On failure decompressed_bytes is None.
    """
    if not raw:
        return None, "Empty content"

    metrics["ocf_marker_count"] = raw.count(OCF_MARKER)

    try:
        cd = int(compression_cd) if compression_cd is not None else None
    except Exception:
        cd = None

    # Uncompressed path
    if cd == 727:
        metrics["decompression_strategy"] = "uncompressed"
        out = strip_ocf_aggressive(raw) if metrics["ocf_marker_count"] > 0 else raw
        _set_length_metrics(out, blob_length, metrics)
        return out, None

    if cd != 728:
        return None, f"Unknown compression: {compression_cd}"

    # Magic-byte sniff after stripping OCF — if the content is already a known
    # container, don't run LZW on it.
    stripped = strip_ocf_aggressive(raw)
    if sniff_container_magic(stripped):
        metrics["decompression_strategy"] = "uncompressed_sniffed"
        _set_length_metrics(stripped, blob_length, metrics)
        return stripped, None

    # LZW cascade: pgodwin port -> ocflzw -> tiff variant -> tolerant variant.
    # Every tier uses `if result:` so that an empty / zero-byte decode is treated
    # as a failure and we fall through to the next tier rather than silently
    # reporting an empty success.
    strategies = [("aggressive", strip_ocf_aggressive), ("conservative", strip_ocf_conservative), ("raw", lambda d: d)]
    last_error = None

    # Tier 1: pgodwin port
    for sname, sfn in strategies:
        try:
            cleaned = sfn(raw)
            result = cerner_lzw_decompress(cleaned)
            if result:
                metrics["decompression_strategy"] = f"pgodwin_port_{sname}"
                _set_length_metrics(result, blob_length, metrics)
                return result, None
        except Exception as e:
            last_error = f"pgodwin_{sname}: {e}"

    # Tier 2: ocflzw library
    if _HAVE_OCFLZW:
        for sname, sfn in strategies:
            try:
                cleaned = sfn(raw)
                result = bytes(LzwDecompress().decompress(cleaned))
                if result:
                    metrics["decompression_strategy"] = f"ocflzw_{sname}"
                    _set_length_metrics(result, blob_length, metrics)
                    return result, None
            except Exception as e:
                last_error = f"ocflzw_{sname}: {e}"

    # Tier 3: TIFF variant
    try:
        result = tiff_lzw_decompress(strip_ocf_aggressive(raw))
        if result:
            metrics["decompression_strategy"] = "tiff_fallback"
            _set_length_metrics(result, blob_length, metrics)
            return result, None
    except Exception as e:
        last_error = f"tiff: {e}"

    # Tier 4: tolerant LSB variant
    try:
        result = tolerant_lzw_decompress(strip_ocf_aggressive(raw))
        if result:
            metrics["decompression_strategy"] = "tolerant_fallback"
            _set_length_metrics(result, blob_length, metrics)
            return result, None
    except Exception as e:
        last_error = f"tolerant: {e}"

    return None, f"LZW failed all tiers: {last_error}"


def _set_length_metrics(decompressed: bytes, blob_length: Optional[int], metrics: dict) -> None:
    if decompressed is None:
        return
    metrics["blob_length_actual"] = len(decompressed)
    metrics["blob_length_expected"] = blob_length
    if blob_length and blob_length > 0:
        diff = abs(len(decompressed) - blob_length) / float(blob_length)
        metrics["blob_length_mismatch"] = bool(diff > BLOB_LENGTH_TOLERANCE)
    else:
        metrics["blob_length_mismatch"] = False

In [0]:
# ==================== MIME DETECTION ====================

try:
    import magic as _magic
    _HAVE_MAGIC = True
except Exception:
    _HAVE_MAGIC = False

try:
    import olefile as _olefile
    _HAVE_OLE = True
except Exception:
    _HAVE_OLE = False


def detect_mime(content: bytes) -> str:
    if not content:
        return 'application/octet-stream'
    if content.startswith(PDF_MAGIC):
        return 'application/pdf'
    if content.startswith(ZIP_MAGIC):
        head = content[:4096]
        if b'word/' in head:
            return 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'
        if b'xl/' in head:
            return 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
        if b'ppt/' in head:
            return 'application/vnd.openxmlformats-officedocument.presentationml.presentation'
        return 'application/zip'
    if content.startswith(OLE_MAGIC):
        return _classify_ole(content)
    if content.startswith(RTF_MAGIC) or content.startswith(RTF_MAGIC_LOOSE):
        return 'text/rtf'
    if _HAVE_MAGIC:
        try:
            return _magic.Magic(mime=True).from_buffer(content) or 'application/octet-stream'
        except Exception:
            pass
    return 'application/octet-stream'


def _classify_ole(data: bytes) -> str:
    if not _HAVE_OLE:
        return 'application/x-ole-storage'
    try:
        with _olefile.OleFileIO(io.BytesIO(data)) as ole:
            if ole.exists('WordDocument'):
                return 'application/msword'
            if ole.exists('Workbook') or ole.exists('Book'):
                return 'application/vnd.ms-excel'
            if ole.exists('PowerPoint Document'):
                return 'application/vnd.ms-powerpoint'
            return 'application/x-ole-storage'
    except Exception:
        return 'application/x-ole-storage'

In [0]:
# ==================== PDF EXTRACTION ====================

try:
    import pypdfium2 as _pdfium
    _HAVE_PDFIUM = True
except Exception:
    _HAVE_PDFIUM = False

try:
    import fitz as _fitz  # PyMuPDF
    _HAVE_FITZ = True
except Exception:
    _HAVE_FITZ = False

try:
    import pytesseract as _pytess
    from PIL import Image as _PILImage
    _HAVE_TESS = True
except Exception:
    _HAVE_TESS = False


def _extract_pdf_pdfium(content: bytes) -> Optional[str]:
    if not _HAVE_PDFIUM:
        return None
    pdf = None
    try:
        pdf = _pdfium.PdfDocument(io.BytesIO(content))
        parts = []
        try:
            for i in range(len(pdf)):
                page = pdf[i]
                tp = None
                try:
                    tp = page.get_textpage()
                    try:
                        t = tp.get_text_range() or ""
                    finally:
                        if tp is not None:
                            try:
                                tp.close()
                            except Exception:
                                pass
                finally:
                    try:
                        page.close()
                    except Exception:
                        pass
                if t.strip():
                    parts.append(t)
        finally:
            # Handled by outer finally below via `pdf.close()`; pass here.
            pass
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if pdf is not None:
            try:
                pdf.close()
            except Exception:
                pass


def _extract_pdf_pymupdf(content: bytes) -> Optional[str]:
    if not _HAVE_FITZ:
        return None
    doc = None
    try:
        doc = _fitz.open(stream=content, filetype="pdf")
        if doc.needs_pass:
            try:
                doc.authenticate("")
            except Exception:
                pass
        parts = []
        for page in doc:
            t = page.get_text("text") or ""
            if t.strip():
                parts.append(t)
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if doc is not None:
            try:
                doc.close()
            except Exception:
                pass


def _extract_pdf_ocr(content: bytes, max_pages: int = OCR_MAX_PAGES, dpi: int = 150) -> Optional[str]:
    if not (_HAVE_FITZ and _HAVE_TESS):
        return None
    if len(content) > OCR_MAX_PDF_SIZE_MB * 1024 * 1024:
        return None
    doc = None
    try:
        doc = _fitz.open(stream=content, filetype="pdf")
        parts = []
        for i, page in enumerate(doc):
            if i >= max_pages:
                break
            try:
                mat = _fitz.Matrix(dpi / 72, dpi / 72)
                pix = page.get_pixmap(matrix=mat)
                img_data = pix.tobytes("png")
                img = _PILImage.open(io.BytesIO(img_data))
                try:
                    # pytesseract's timeout= forwards to the tesseract subprocess
                    # and actually kills it; prevents pathological PDFs from hanging.
                    page_text = _pytess.image_to_string(
                        img, lang=OCR_LANG, timeout=OCR_PAGE_TIMEOUT_SEC
                    )
                except RuntimeError:
                    # pytesseract raises RuntimeError when the tesseract
                    # subprocess exceeds timeout
                    continue  # skip this page, move to the next
                if page_text and page_text.strip():
                    parts.append(page_text.strip())
            except Exception:
                continue
        return "\n\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if doc is not None:
            try:
                doc.close()
            except Exception:
                pass

In [0]:
# ==================== DOCX / HTML / RTF / EXCEL ====================

try:
    import mammoth as _mammoth
    _HAVE_MAMMOTH = True
except Exception:
    _HAVE_MAMMOTH = False

try:
    import docx2txt as _docx2txt
    _HAVE_DOCX2TXT = True
except Exception:
    _HAVE_DOCX2TXT = False

try:
    from resiliparse.parse.html import HTMLTree as _RespHTMLTree
    from resiliparse.extract.html2text import extract_plain_text as _resp_extract
    _HAVE_RESILIPARSE = True
except Exception:
    _HAVE_RESILIPARSE = False

try:
    from striprtf.striprtf import rtf_to_text as _rtf_to_text
    _HAVE_STRIPRTF = True
except Exception:
    _HAVE_STRIPRTF = False

try:
    from openpyxl import load_workbook as _openpyxl_load
    _HAVE_OPENPYXL = True
except Exception:
    _HAVE_OPENPYXL = False


def _extract_docx(content: bytes) -> Optional[str]:
    # Primary: mammoth (preserves headings/bullets/tables)
    if _HAVE_MAMMOTH:
        try:
            res = _mammoth.convert_to_markdown(io.BytesIO(content))
            if res.value and res.value.strip():
                return res.value
        except Exception:
            pass
    # Fallback: docx2txt
    if _HAVE_DOCX2TXT:
        try:
            return _docx2txt.process(io.BytesIO(content))
        except Exception:
            pass
    return None


def _extract_html(content: bytes) -> Optional[str]:
    if not _HAVE_RESILIPARSE:
        # Fall back to BS4 (v1 behaviour) if resiliparse is missing.
        return _extract_html_bs4(content)
    for enc in ('utf-8', 'windows-1252', 'latin-1'):
        try:
            html = content.decode(enc, errors='ignore')
            if '<' not in html or '>' not in html:
                continue
            tree = _RespHTMLTree.parse(html)
            text = _resp_extract(tree, preserve_formatting='minimal_html', main_content=False)
            if text and text.strip():
                return text
        except Exception:
            continue
    return None


def _extract_html_bs4(content: bytes) -> Optional[str]:
    try:
        from bs4 import BeautifulSoup
    except Exception:
        return None
    for enc in ('utf-8', 'windows-1252', 'latin-1'):
        try:
            html = content.decode(enc, errors='ignore')
            if '<' not in html or '>' not in html:
                continue
            soup = BeautifulSoup(html, 'html.parser')
            for element in soup(['script', 'style', 'head', 'title', 'meta']):
                element.decompose()
            text = soup.get_text(separator=' ', strip=True)
            if text:
                return text
        except Exception:
            continue
    return None


def _extract_rtf(content: bytes) -> Optional[str]:
    if not (_HAVE_STRIPRTF and content):
        return None
    # Report: Cerner RTF is always cp1252. Skip UTF-8 trial.
    try:
        rtf_text = content.decode('cp1252', errors='ignore')
        if not (rtf_text.startswith('{\\rtf') or rtf_text.startswith('{\\')):
            # Second chance: utf-8 for non-Cerner RTFs
            rtf_text = content.decode('utf-8', errors='ignore')
            if not (rtf_text.startswith('{\\rtf') or rtf_text.startswith('{\\')):
                return None
        plain = _rtf_to_text(rtf_text, errors='ignore')
        return plain if plain and plain.strip() else None
    except Exception:
        return None


def _extract_excel(content: bytes) -> Optional[str]:
    if not _HAVE_OPENPYXL:
        return None
    try:
        wb = _openpyxl_load(io.BytesIO(content), read_only=True, data_only=True)
        parts = []
        for sheet in wb.sheetnames:
            ws = wb[sheet]
            for row in ws.iter_rows(values_only=True):
                row_text = ' '.join(str(c) for c in row if c is not None)
                if row_text.strip():
                    parts.append(row_text)
        return '\n'.join(parts) if parts else None
    except Exception:
        return None


def _guess_text(content: bytes) -> Tuple[Optional[str], Optional[str]]:
    # Lightweight encoding detect; report recommends short-circuiting known formats.
    # Here we only handle genuinely unknown binary-ish content.
    if not content:
        return None, None
    # BOM checks — decode strictly; BOM is a strong signal and a UnicodeDecodeError
    # here means the stream is genuinely malformed for that encoding, so fall through.
    if content.startswith(b'\xff\xfe'):
        try:
            return content.decode('utf-16-le'), 'utf-16-le'
        except UnicodeDecodeError:
            pass
    if content.startswith(b'\xfe\xff'):
        try:
            return content.decode('utf-16-be'), 'utf-16-be'
        except UnicodeDecodeError:
            pass
    if content.startswith(b'\xef\xbb\xbf'):
        try:
            return content[3:].decode('utf-8'), 'utf-8-sig'
        except UnicodeDecodeError:
            pass
    # Strict utf-8: only succeeds on well-formed UTF-8. Falls through on invalid
    # byte sequences so cp1252/latin-1 get a real chance (the previous
    # errors='ignore' path always succeeded and made this cascade dead code).
    try:
        return content.decode('utf-8'), 'utf-8'
    except UnicodeDecodeError:
        pass
    # Strict cp1252: almost all byte sequences decode, but some (0x81/0x8D/0x8F/
    # 0x90/0x9D) are undefined and raise — fall through to latin-1 replace.
    try:
        return content.decode('cp1252'), 'cp1252'
    except UnicodeDecodeError:
        pass
    # latin-1 (ISO-8859-1) can't fail (every byte is a valid code point), but we
    # use errors='replace' so any truly undecodable points would surface as U+FFFD
    # rather than silently disappear.
    return content.decode('latin-1', errors='replace'), 'latin-1'

In [0]:
# ==================== PARSE DISPATCHER ====================

def parse(content: bytes, provided_type=None) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """Returns (text, content_type, encoding).
    For binary content types that are not text, returns (None, content_type, None).
    """
    if not content:
        return None, None, None
    content_type = provided_type or detect_mime(content)

    if content_type in BINARY_CONTENT_TYPES:
        return None, content_type, None

    if content_type == 'application/pdf':
        t = _extract_pdf_pdfium(content) or _extract_pdf_pymupdf(content)
        if t:
            return t, content_type, 'utf-8'
        t = _extract_pdf_ocr(content)
        if t:
            return t, content_type, 'utf-8'
        # Give a concrete status string so callers don't mistake this for a crash
        return "[PDF - extraction failed]", content_type, None

    if content_type == 'application/vnd.openxmlformats-officedocument.wordprocessingml.document':
        t = _extract_docx(content)
        if t:
            return t, content_type, 'utf-8'
        # DOCX is a ZIP container — do NOT fall through to _guess_text on the
        # raw ZIP bytes or we'd produce garbled "text" that looks legitimate.
        return "[DOCX - extraction failed]", content_type, 'utf-8'

    if content_type in ('application/vnd.ms-excel',
                        'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'):
        t = _extract_excel(content)
        if t:
            return t, content_type, 'utf-8'
        # xlsx is a ZIP, xls is OLE2 — both binary containers; never _guess_text.
        return "[Excel - extraction failed]", content_type, 'utf-8'

    if content_type in ('text/html', 'text/xml', 'application/xhtml+xml'):
        t = _extract_html(content)
        if t:
            return t, content_type, 'utf-8'
        # Both resiliparse and BS4 extractors failed; don't _guess_text raw HTML.
        return "[HTML - extraction failed]", content_type, 'utf-8'

    if content_type == 'text/rtf' or content.startswith(RTF_MAGIC) or content.startswith(RTF_MAGIC_LOOSE):
        t = _extract_rtf(content)
        if t:
            return t, (content_type or 'text/rtf'), 'utf-8'
        # Consistent with Fix 2 pattern: structured RTF that couldn't be parsed
        # becomes a failure marker rather than falling through to _guess_text,
        # which would spit out raw RTF control words as "text".
        return "[RTF - extraction failed]", (content_type or 'text/rtf'), 'utf-8'

    # Plaintext / unknown
    decoded, enc = _guess_text(content)
    if decoded:
        return decoded, content_type, enc

    return "[Binary data, unable to decode]", content_type, None

In [0]:
# ==================== PARSE HANG DEFENSE ====================
# The OCR subprocess timeout (OCR_PAGE_TIMEOUT_SEC) only covers tesseract; the
# pypdfium2 / PyMuPDF primary text extraction paths can themselves hang on
# pathological PDFs with no Python-level timeout API. Size cap short-circuits
# the worst cases (GB-scale decompression DoS) and the subprocess timeout
# enforces a real wall-clock limit on the whole parse() call.

# Use loky's ProcessPoolExecutor (not concurrent.futures') — loky serializes
# submitted tasks via cloudpickle, which correctly handles functions defined
# in %run-included notebooks (where stdlib pickle fails with PicklingError).
# Loky also defaults to spawn-mode subprocess, which is safer than fork inside
# a PySpark Python worker (avoids inheriting JVM-bridge/MKL/OpenMP state).
from loky import ProcessPoolExecutor
from concurrent.futures import TimeoutError as _FutureTimeout


def _is_probable_pdf(content: bytes, provided_type: Optional[str]) -> bool:
    if provided_type and "pdf" in provided_type.lower():
        return True
    return bool(content) and content[:4] == b"%PDF"


class ParsePool:
    """Reusable subprocess pool for bounded-time parse() calls.

    Usage inside an executor pandas UDF:
        with ParsePool() as pool:
            for row in ...:
                text, ct, enc, parse_status = pool.parse(content, provided_type)

    On timeout the pool is torn down and recreated (the worker subprocess may
    still be running native C code that can't be interrupted Python-side).
    Uses loky.ProcessPoolExecutor internally (cloudpickle-aware, spawn-mode
    default). Safe to construct inside a PySpark Python worker.
    """

    def __init__(self, timeout: int = PARSE_TIMEOUT_SEC):
        self._timeout = timeout
        self._pool: Optional[ProcessPoolExecutor] = None

    def __enter__(self):
        self._pool = ProcessPoolExecutor(max_workers=1)
        return self

    def __exit__(self, *a):
        self._teardown()

    def _teardown(self):
        if self._pool is not None:
            try:
                self._pool.shutdown(wait=False, cancel_futures=True)
            except Exception:
                pass
            self._pool = None

    def _recreate(self):
        self._teardown()
        self._pool = ProcessPoolExecutor(max_workers=1)

    def parse(self, content: bytes, provided_type: Optional[str] = None):
        """Returns (text, content_type, encoding, parse_status)
        where parse_status is one of: "ok", "pdf_too_large", "timeout", "error:<class>".
        """
        if _is_probable_pdf(content, provided_type) and len(content) > PDF_EXTRACT_MAX_BYTES:
            return "[PDF - too large]", "application/pdf", "utf-8", "pdf_too_large"
        if self._pool is None:
            self._pool = ProcessPoolExecutor(max_workers=1)
        try:
            fut = self._pool.submit(parse, content, provided_type)
            text, ct, enc = fut.result(timeout=self._timeout)
            return text, ct, enc, "ok"
        except _FutureTimeout:
            self._recreate()
            return ("[PARSE - timeout]", provided_type or "application/octet-stream",
                    "utf-8", "timeout")
        except Exception as e:
            # Runtime error inside the child; pool may be poisoned
            self._recreate()
            return (f"[PARSE - error: {type(e).__name__}]",
                    provided_type or "application/octet-stream",
                    "utf-8", f"error:{type(e).__name__}")


def parse_with_timeout(content: bytes, provided_type: Optional[str] = None,
                       timeout: int = PARSE_TIMEOUT_SEC):
    """Single-shot convenience wrapper (creates and tears down a pool).

    Prefer ParsePool for batches — avoids fork-per-row overhead.
    """
    with ParsePool(timeout=timeout) as pool:
        return pool.parse(content, provided_type)

In [0]:
# ==================== POST-PROCESSOR ====================

import ftfy as _ftfy
from ftfy import TextFixerConfig as _FtfyConfig

_FTFY_CONFIG = _FtfyConfig(
    fix_c1_controls=True,
    restore_byte_a0=True,
    fix_line_breaks=True,
    normalization="NFC",
    fix_latin_ligatures=False,
)

# Garbage patterns carried over from v1
_GARBAGE_PATTERNS = [
    r'f{8,}',
    r'(f5ebe8){2,}',
    r'(bec7c2){2,}',
    r'(333333){2,}',
    r'(cdcdcd){2,}',
    r'(elnlueve){2,}',
    r'(7f){4,}',
]
_GARBAGE_REGEX = re.compile('|'.join(_GARBAGE_PATTERNS), re.IGNORECASE)
_OCF_TEXT_PATTERN = re.compile(r'\s*ocf_blob\s*', re.IGNORECASE)
_TEMPLATE_PATTERN = re.compile(r'<%[A-Z_]*%>', re.IGNORECASE)


def _strip_ocf_text(text: str) -> str:
    if not text or 'ocf_blob' not in text.lower():
        return text
    return _OCF_TEXT_PATTERN.sub('', text).strip()


def _strip_template(text: str) -> str:
    if not text or '<%' not in text:
        return text
    return _TEMPLATE_PATTERN.sub('', text).strip()


def _truncate_at_garbage(text: str) -> Optional[str]:
    if not text:
        return text
    m = _GARBAGE_REGEX.search(text)
    if not m:
        return text
    pos = m.start()
    if pos < 20:
        return None
    truncated = text[:pos].rstrip()
    truncated = re.sub(r'[0-9a-f]{6,}$', '', truncated, flags=re.IGNORECASE).rstrip()
    return truncated if len(truncated) >= 20 else None


def _detect_truncation(text: str, content_type: str, decompressed_bytes: bytes,
                       blob_length: Optional[int], decompressor_flag: bool) -> Tuple[bool, Optional[str]]:
    """Returns (truncation_flag, truncation_reason)."""
    if decompressor_flag:
        return True, "blob_length_mismatch"
    if content_type == 'text/rtf' and text:
        # Brace-count mismatch is the reliable RTF truncation signal. The
        # previous last-char check (flag if stripped end is not '}' or '.')
        # over-triggered on healthy RTF that ends with \par, a control word,
        # or plain text — after extraction those are all normal endings.
        if text.count('{') != text.count('}'):
            return True, "rtf_brace_mismatch"
    if content_type == 'application/pdf' and decompressed_bytes:
        tail = decompressed_bytes[-1024:]
        if b'%%EOF' not in tail:
            return True, "pdf_no_eof"
    if text and decompressed_bytes:
        dlen = len(decompressed_bytes)
        # Catch truncation on *both* sides of an 8192-byte boundary: LZW can
        # stop mid-code-block at e.g. 8190 or 16386 bytes, so accept remainder
        # within 64 of either edge. (abs(dlen % 8192) is redundant — modulo
        # is already non-negative.)
        if dlen > 8192:
            rem = dlen % 8192
            if rem < 64 or rem > 8128:
                last = text.rstrip()
                if last and last[-1].isalpha():
                    return True, "ends_mid_word_near_8192"
    return False, None


def post_process(text: str, content_type: str, decompressed_bytes: bytes,
                 blob_length: Optional[int], decompressor_flag: bool = False
                 ) -> Tuple[Optional[str], dict]:
    """Clean text; return (cleaned_text, metadata).

    metadata keys: ftfy_explain (str JSON | None), ftfy_failed (bool),
    truncation_flag (bool), truncation_reason (str | None)
    """
    meta = {"ftfy_explain": None, "ftfy_failed": False,
            "truncation_flag": False, "truncation_reason": None}
    if not isinstance(text, str) or not text:
        return text, meta

    # 1. OCF marker strip
    text = _strip_ocf_text(text)
    if not text:
        return None, meta

    # 2. Template marker strip
    text = _strip_template(text)
    if not text:
        return None, meta

    # 3. Garbage truncate (can return None if almost all garbage)
    text = _truncate_at_garbage(text)
    if not text:
        return None, meta

    # 4. ftfy with audit. Split the json.dumps serialization into its own
    # try/except so a non-serializable step can't silently bypass the fallback
    # or corrupt `text`. Any ftfy *or* serialization failure flips
    # meta["ftfy_failed"] = True so the regression is observable downstream.
    try:
        fixed, explanation = _ftfy.fix_and_explain(text, config=_FTFY_CONFIG)
        text = fixed
        if explanation:
            try:
                meta["ftfy_explain"] = json.dumps(
                    [(step.transformation, step.encoding) if hasattr(step, 'transformation') else str(step)
                     for step in explanation]
                )
            except Exception:
                meta["ftfy_failed"] = True
                meta["ftfy_explain"] = None
    except Exception:
        # Never block on ftfy; fall back to v1-style regex replacement
        meta["ftfy_failed"] = True
        for bad, good in [('Â ', ' '), ('â€™', "'"), ('Ã©', 'é')]:
            text = text.replace(bad, good)

    # 5. Final whitespace cleanup
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    if len(text) < 1:
        return None, meta

    # 6. Truncation heuristic
    flag, reason = _detect_truncation(text, content_type, decompressed_bytes, blob_length, decompressor_flag)
    meta["truncation_flag"] = flag
    meta["truncation_reason"] = reason

    return text, meta


print(f"blob_shared_lib loaded: decompressor={DECOMPRESSOR_VERSION} parser={PARSER_VERSION} post_processor={POST_PROCESSOR_VERSION}")